In [1]:
import rectanglepy as rectangle
import pandas as pd
from anndata import AnnData
import h5py
import spatialdata_io
import helper
import biomart
from spatialdata import SpatialData

In [ ]:


counts = pd.read_csv('../data/human_m1/matrix.csv', index_col=0)
counts = counts.astype(int)

In [7]:

metadata = pd.read_csv('../data/human_m1/metadata.csv', index_col=0)


In [14]:
annotations = metadata['subclass_label']

In [15]:
set_annotations = set(list(annotations))

In [ ]:

# remove anything after the second space(word)
annotations = annotations.str.split(' ', n=2).str[:2].str.join(' ')
# remove anything afert first minus
annotations = annotations.str.split('-', n=1).str[0]

In [ ]:
annotations.value_counts()

In [ ]:
adata = AnnData(counts, obs=annotations.to_frame(name='cell_type'))

In [18]:
adata.write_h5ad('../data/human_m1/allen_brain.h5ad')

In [2]:
f = spatialdata_io.visium(path='../data', dataset_id='151673',scalefactors_file='scalefactors_json.json', tissue_positions_file='tissue_positions.csv', counts_file='151673_filtered_feature_bc_matrix.h5')


INFO     reading ../data/151673_filtered_feature_bc_matrix.h5                                                      


/Users/berni/PycharmProjects/rectangle_paper/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/berni/PycharmProjects/rectangle_paper/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/berni/PycharmProjects/rectangle_paper/.venv/lib/python3.12/site-packages/spatialdata/models/models.py:1144: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)


In [6]:
f.write('../data/spatialdata_spatial.zarr')

INFO     The Zarr backing store has been changed from None the new file path: ../data/spatialdata_spatial.zarr     


In [12]:
import spatialdata as sd
f_zarr = sd.read_zarr('../data/spatialdata_spatial.zarr')


In [3]:
data_table = f['table']
bulks = data_table.to_df()
# Convert bulks from counts to CPM
bulks_cpm = bulks.div(bulks.sum(axis=1), axis=0) * 1e6

In [4]:
bulks_cpm.to_csv('../data/spatial_bulks_cpm.csv')

In [5]:
bulks_cpm_csv = pd.read_csv('../data/spatial_bulks_cpm.csv', index_col=0)

In [ ]:
estimations, signature_result = rectangle.rectangle(adata, bulks_cpm)

In [20]:
estimations.to_csv('../output/spatial_estimations.csv')

In [21]:
import pickle
with open('../output/signature_result_spatial.pkl', 'wb') as f:
    pickle.dump(signature_result, f)

In [30]:
from spatialdata.models import TableModel

estimations = pd.read_csv('../output/spatial_estimations.csv', index_col=0)

# Create an AnnData object from the estimations

table_data = AnnData(estimations)
adata_for_sdata = TableModel.parse(table_data)

adata_for_sdata.uns["spatialdata_attrs"] = {
    "region": "spots",
    "region_key": "region",  # column in adata.obs that will link a given obs to the elements it annotates
    "instance_key": "spot_id",  # column that matches a given obs in the table to a given circle
}

adata_for_sdata.obs["region"] = pd.Categorical(["spots"] * len(adata_for_sdata))
adata_for_sdata.obs["spot_id"] = f.tables.data['table'].obs["spot_id"]

f.tables["rectangle_results"] = adata_for_sdata





/Users/berni/PycharmProjects/rectangle_paper/.venv/lib/python3.12/site-packages/spatialdata/_core/_elements.py:125: UserWarning: Key `rectangle_results` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


In [2]:
estimations = pd.read_csv('../output/spatial_estimations.csv', index_col=0)


In [3]:
estimations


,Astro L1,Endo L2,Exc L2,Exc L3,Exc L5,Exc L6,Inh L1,Inh L2,Inh L3,Inh L5,Inh L6,Micro L1,OPC L1,Oligo L2,Oligo L3,Oligo L5,VLMC L1,Unknown
AAACAAGTATCTCCCA-1,0.212367,9.101038e-02,1.950827e-01,3.148337e-01,-5.217918e-18,1.380020e-01,4.410832e-03,-2.083133e-18,-6.717407e-18,-3.635367e-18,1.606721e-02,9.511625e-20,2.619346e-04,6.709796e-03,6.350777e-19,0.021255,-5.476314e-19,0.0
AAACAATCTACTAGCA-1,0.406569,5.947108e-02,5.485971e-04,8.659744e-04,-6.166916e-18,2.663651e-01,3.598131e-03,-2.962296e-18,-6.546982e-18,-2.482936e-18,7.194467e-02,-4.615099e-19,-1.900484e-18,2.486857e-02,1.578029e-18,0.163005,2.763078e-03,0.0
AAACACCAATAACTGC-1,0.186169,-2.102301e-19,-7.995295e-19,3.647418e-17,8.669836e-18,1.648704e-04,9.531228e-20,-5.507192e-19,-1.370423e-17,5.594861e-18,-5.063611e-20,2.242993e-19,2.754422e-04,3.970003e-01,2.641885e-01,0.152201,1.705212e-19,0.0
AAACAGAGCGACTCCT-1,0.520464,6.322071e-02,2.962754e-01,-1.001170e-18,-3.700719e-18,5.273599e-04,-6.394109e-20,5.462067e-19,2.055084e-02,4.392329e-18,3.939512e-02,4.040248e-04,6.294595e-05,5.151464e-02,3.870951e-19,0.007585,1.293190e-19,0.0
AAACAGCTTTCAGAAG-1,0.195324,2.428308e-02,7.225817e-02,-3.993920e-18,-3.005352e-18,3.610746e-01,6.545068e-04,-3.404500e-19,-8.674508e-20,1.069185e-18,2.687012e-02,-1.943455e-19,-1.397410e-19,7.804248e-02,7.093159e-18,0.201725,3.976865e-02,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTGTTTCACATCCAGG-1,0.087417,8.086133e-03,-3.148038e-18,2.323289e-18,-2.042705e-19,1.733212e-18,-1.838735e-19,-3.354900e-19,7.920311e-18,-2.384789e-18,-1.889328e-19,1.564176e-18,1.718673e-19,6.132820e-01,2.707533e-01,0.020461,-2.122717e-20,0.0
TTGTTTCATTAGTCTA-1,0.434722,1.764729e-03,-9.657652e-19,-4.364779e-18,3.738030e-18,1.333245e-19,1.061108e-18,1.186719e-18,4.411898e-18,-1.309014e-17,-9.070593e-20,4.503926e-19,3.090895e-20,6.518974e-02,2.583968e-01,0.239927,-1.250325e-19,0.0
TTGTTTCCATACAACT-1,0.365246,1.217998e-01,-4.558343e-19,-6.776007e-19,-1.074339e-18,1.180285e-01,-1.791171e-19,1.096178e-18,-2.758537e-18,5.902919e-18,1.648008e-02,4.910707e-03,-1.359879e-19,6.806299e-03,-7.863117e-18,0.366729,7.226882e-19,0.0
TTGTTTGTATTACACG-1,0.397432,-1.360867e-20,5.655527e-04,-1.211893e-17,2.543319e-03,1.123088e-01,1.589010e-19,4.061507e-02,7.139095e-18,-4.785501e-18,1.503287e-05,1.871296e-04,5.257131e-20,9.277203e-19,-1.915094e-18,0.446333,5.678090e-20,0.0


In [4]:
base = estimations.columns.str.replace(r'\s+L\d+$', '', regex=True)
summed = estimations.T.groupby(base).sum().T

In [5]:
summed.to_csv('../output/spatial_estimations_summed.csv')